# 2.2 State — Custom Agent State Fields

## What you will learn in this notebook

- What **LangGraph state** is and how it differs from messages and context
- How to **extend the default agent state** with custom typed fields
- How tools can **write to state** using `Command`
- How tools can **read from state** using `ToolRuntime`
- How to **pre-populate state** at invocation time

---

### What is state in LangGraph?

Every LangGraph agent runs inside a **graph** that carries a **state dict** as it executes. The default state only has one field:
- `messages` — the conversation history list

**Custom state** lets you add your own fields to this dict — typed data that:
- Persists across turns within a thread (saved by the checkpointer)
- Can be written by tools and read by other tools in later turns
- Is invisible to the user (not in the conversation)

### State vs Runtime Context — the key difference

| | State | Runtime Context |
|---|---|---|
| **Set by** | Tools during the conversation | You at call time (server-side) |
| **Persists** | Yes — saved by checkpointer | No — fresh per call |
| **Changes over time** | Yes — tool can update it | No — read-only per request |
| **Best for** | Data the agent discovers/computes | Per-user profile data you already have |

### The `Command` pattern

When a tool wants to update state, it returns a `Command` object instead of a plain value. `Command` tells LangGraph: *"run this update to the state graph in addition to returning a tool result"*.

In [1]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# ============================================================
# CELL 2: Define a Custom State Schema
# ============================================================
# AgentState is LangGraph's base state class — it already includes
# the 'messages' field. We extend it by subclassing.
#
# Adding 'favourite_colour: str' means:
#   - The state dict now has a 'favourite_colour' key
#   - It is typed (str) and validated by LangGraph
#   - It is persisted by the checkpointer alongside 'messages'
#   - It can be read and written by tools in any turn of the conversation
#
# In a real app this might be:
#   class UserSessionState(AgentState):
#       user_intent: str           # Detected intent (booking, refund, etc.)
#       items_in_cart: list[str]   # Items added during the session
#       discount_applied: bool     # Whether a coupon was used

from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str   # Our custom field — persists across turns

---
## Section 1 — Writing to State From a Tool

In [3]:
# ============================================================
# CELL 3: Define a Tool That Updates State
# ============================================================
# A tool that writes to state must return a Command object
# instead of a plain value.
#
# Command(update={...}) tells LangGraph:
#   "Merge this dict into the graph state"
#
# The update dict here has TWO keys:
#
#   "favourite_colour": favourite_colour
#     → Writes the user's colour to our custom state field.
#       This value persists across turns for the same thread_id.
#
#   "messages": [ToolMessage(...)]
#     → REQUIRED when using Command — we must manually add the
#       ToolMessage to the messages list. Normally LangChain does
#       this automatically, but Command bypasses that, so we do it
#       ourselves. tool_call_id links this message to the AI's
#       tool_call request (so LangGraph knows which call this answers).
#
# runtime: ToolRuntime provides runtime.tool_call_id — the unique
# ID of the tool call that triggered this tool, needed for the
# ToolMessage linkage.

from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour,   # Write to custom state field
        "messages": [                            # Must manually include ToolMessage
            ToolMessage(
                "Successfully updated favourite colour",
                tool_call_id=runtime.tool_call_id  # Links to the AI's tool call request
            )
        ]
    })

In [6]:
# ============================================================
# CELL 4: Create the Agent With Custom State Schema
# ============================================================
# state_schema=CustomState tells LangGraph to use our extended
# state class instead of the default AgentState.
# This enables the 'favourite_colour' field in the state dict.
#
# checkpointer=InMemorySaver() saves the full state (including
# favourite_colour) between turns — so the update persists.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "groq:qwen/qwen3-32b",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState        # Use our extended state class
)

In [7]:
# ============================================================
# CELL 5: Tell the Agent Your Favourite Colour
# ============================================================
# The agent will:
#   1. Read "My favourite colour is green"
#   2. Recognise this matches update_favourite_colour's purpose
#   3. Call update_favourite_colour(favourite_colour="green")
#   4. Tool returns Command(update={"favourite_colour": "green", ...})
#   5. LangGraph merges the update into state
#   6. Agent confirms the update
#
# After this call, the state for thread_id="1" contains:
#   {"messages": [...], "favourite_colour": "green"}

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}  # State is saved under thread "1"
)

In [8]:
# ============================================================
# CELL 6: Inspect — See favourite_colour in the Response
# ============================================================
# The response dict now has TWO keys:
#   response['messages']          → conversation history (as always)
#   response['favourite_colour']  → "green"  ← our custom state field!
#
# This is what state looks like: a first-class field on the response,
# not buried in a message or tool call.

from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='fd70366c-0b41-44b1-a326-dcab17eff497'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user said their favorite color is green. I need to update the state with that information. There\'s a function called update_favourite_colour that takes a parameter favourite_colour. So I should call that function with the value "green". Let me make sure the parameters are correct. The function requires a string, and the user provided a color name, which is a string. Alright, I\'ll generate the tool call with the color green.\n', 'tool_calls': [{'id': 'yvxa01der', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 166, 'total_tokens': 284, 'completion_time': 0.233237071, 'compl

In [9]:
# ============================================================
# CELL 7: Pre-Populate State at Invocation Time
# ============================================================
# You can also SET state directly in the input dict rather than
# having a tool update it. This is useful when you already know
# the value server-side and want to seed the state before the
# conversation starts.
#
# Here thread_id="10" is a fresh conversation, so state is empty.
# We pass "favourite_colour": "green" directly in the input.
# LangGraph merges this into the initial state for this thread.
#
# This is useful for pre-loading user preferences from a database:
#   user = db.get_user(user_id)
#   agent.invoke({"messages": [...], "favourite_colour": user.fav_colour})

response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"   # Pre-populate state at call time
    },
    {"configurable": {"thread_id": "10"}}  # New thread — seeded with the value
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='1178e6eb-ceb2-45fd-b409-2d576ff5ef78'),
              AIMessage(content="Hi! I'm doing well. How about you?", additional_kwargs={'reasoning_content': 'Okay, the user asked "Hello, how are you?" which is a greeting. I need to respond appropriately. Since there\'s no function provided for handling greetings or general conversation, I should just reply politely. Let me check the tools again to make sure I didn\'t miss anything. The only function available is update_favourite_colour, which requires the user to reveal their favorite color. The user hasn\'t mentioned their color preference here, so I don\'t need to use the function. A simple friendly response like "Hi! I\'m doing well. How about you?" should be suitable here.\n'}, response_metadata={'token_usage': {'completion_tokens': 135, 'prompt_tokens': 167, 'total_tokens': 302, 'completion_time': 0.2716

---
## Section 2 — Reading State From a Tool

In [11]:
# ============================================================
# CELL 8: Define a Tool That Reads From State
# ============================================================
# runtime.state is a dict containing the current full state.
# We use runtime.state["favourite_colour"] to read the field.
#
# Defensive programming: we wrap it in try/except for KeyError.
# If the user hasn't set their colour yet, the field may not exist
# in state — accessing a missing key would crash the tool.
# Returning a descriptive string lets the agent handle the missing
# data gracefully in natural language.
#
# Note the difference:
#   runtime.context.field  → reads from runtime CONTEXT (injected at call time, read-only)
#   runtime.state["field"] → reads from graph STATE (persisted, can be written)

@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]   # Read from persisted state
    except KeyError:
        return "No favourite colour found in state"  # Graceful fallback

# Recreate agent with BOTH update and read tools
agent = create_agent(
    "groq:qwen/qwen3-32b",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [12]:
# ============================================================
# CELL 9: Turn 1 — Set the Colour (Writes to State)
# ============================================================
# The agent calls update_favourite_colour("green").
# State for thread "1" now has favourite_colour="green".

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='fc2220c4-b68e-435e-bc3a-d4b43e2b6411'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user just told me their favorite color is green. Let me check the available tools. There\'s an update_favourite_colour function that takes a favourite_colour parameter. Since they provided "green", I should call that function to update their state. No need to use the read function here because the user already stated their preference. Just need to make sure the arguments are correctly formatted as JSON. Alright, I\'ll generate the tool call with the color green.\n', 'tool_calls': [{'id': 'zdvy70ekp', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 213, 'total_tokens': 334, '

In [13]:
# ============================================================
# CELL 10: Turn 2 — Ask About the Colour (Reads From State)
# ============================================================
# Same thread_id="1" — the checkpointer loads the state that was
# saved in Turn 1 (favourite_colour="green").
#
# The agent calls read_favourite_colour().
# runtime.state["favourite_colour"] → "green".
# Agent replies: "Your favourite colour is green."
#
# This is different from conversation memory:
#   Memory: the agent re-reads the message "My favourite colour is green"
#   State:  the agent calls a tool that reads a structured, typed field
#           — no need to re-parse natural language

response = agent.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}  # Same thread — state is loaded
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='fc2220c4-b68e-435e-bc3a-d4b43e2b6411'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user just told me their favorite color is green. Let me check the available tools. There\'s an update_favourite_colour function that takes a favourite_colour parameter. Since they provided "green", I should call that function to update their state. No need to use the read function here because the user already stated their preference. Just need to make sure the arguments are correctly formatted as JSON. Alright, I\'ll generate the tool call with the color green.\n', 'tool_calls': [{'id': 'zdvy70ekp', 'function': {'arguments': '{"favourite_colour":"green"}', 'name': 'update_favourite_colour'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 213, 'total_tokens': 334, '

---
## Summary

| Concept | Key takeaway |
|---|---|
| **`AgentState` subclass** | Extend default state by adding typed fields to a subclass |
| **`state_schema=`** | Pass your subclass to `create_agent()` to enable custom fields |
| **`Command(update={...})`** | Return from a tool to write to state — must include a ToolMessage |
| **`runtime.state["field"]`** | Read from state inside a tool via ToolRuntime |
| **Pre-populate state** | Pass custom fields in the input dict to seed state at invocation |
| **Checkpointer required** | State persists across turns only if `checkpointer=` is set |

### State vs Memory vs Context — final comparison

```
Messages (memory):
  "My favourite colour is green"  →  stored in conversation history
  Agent re-reads natural language to remember

Custom State:
  update_favourite_colour("green")  →  stored in state["favourite_colour"]
  Agent reads typed field — no natural language parsing needed

Runtime Context:
  invoke(context=UserContext(fav_colour="green"))  →  injected per request
  Not persisted — fresh every call
```

### Next steps
- **2.3 Multi-Agent** — building systems where a main agent delegates to specialised sub-agents

In [9]:
from langchain.agents import AgentState, create_agent
from langchain_core.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool, ToolRuntime
from typing import Any


# ── Custom State ───────────────────────────────────────────────────────────────

class CustomState(AgentState):
    demographics: dict[str, Any]
    preferences: dict[str, Any]


# ── Generic Tools (reusable across any agent) ─────────────────────────────────
# The LLM calls these naturally based on conversation context.
# Tool docstrings are what guide the LLM — not the system prompt.

from typing import Any

@tool
def store_user_demographics(
    demographics: dict[str, Any],   # was dict[str, dict[str, str]] — too strict
    runtime: ToolRuntime
) -> Command:
    """
    Store personal details about the user such as name, age, location, occupation, etc.
    Pass a flat or nested dict e.g. {"name": "Prudhvi", "city": "Rajahmundry", "age": "34"}.
    """
    print("[tool] storing demographics:", demographics)
    return Command(update={
        "demographics": demographics,
        "messages": [
            ToolMessage(
                "User demographics updated successfully.",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })


@tool
def store_user_preferences(
    preferences: dict[str, Any],    # relaxed for same reason
    runtime: ToolRuntime
) -> Command:
    """
    Store the user's preferences such as cuisine type, dietary restrictions,
    spice level, language, or any other personal preference.
    Pass a flat dict e.g. {"cuisine": "Italian", "spice_level": "mild"}.
    """
    print("[tool] storing preferences:", preferences)
    return Command(update={
        "preferences": preferences,
        "messages": [
            ToolMessage(
                "User preferences updated successfully.",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })


@tool
def store_user_preferences(
    preferences: dict[str, Any],    # relaxed for same reason
    runtime: ToolRuntime
) -> Command:
    """
    Store the user's preferences such as cuisine type, dietary restrictions,
    spice level, language, or any other personal preference.
    Pass a flat dict e.g. {"cuisine": "Italian", "spice_level": "mild"}.
    """
    print("[tool] storing preferences:", preferences)
    return Command(update={
        "preferences": preferences,
        "messages": [
            ToolMessage(
                "User preferences updated successfully.",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })


@tool
def get_user_demographics(runtime: ToolRuntime) -> Command:
    """
    Retrieve previously stored personal details about the user.
    Use this when you need context about who the user is.
    """
    print("[tool] fetching demographics")
    demographics = runtime.state.get("demographics", {})
    return Command(update={
        "messages": [
            ToolMessage(
                str(demographics) if demographics else "No demographics stored yet.",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })


@tool
def get_user_preferences(runtime: ToolRuntime) -> Command:
    """
    Retrieve previously stored preferences of the user.
    Use this when personalising recommendations or responses.
    """
    print("[tool] fetching preferences")
    preferences = runtime.state.get("preferences", {})
    return Command(update={
        "messages": [
            ToolMessage(
                str(preferences) if preferences else "No preferences stored yet.",
                tool_call_id=runtime.tool_call_id
            )
        ]
    })


# ── Agent ─────────────────────────────────────────────────────────────────────
# System prompt is about the agent's PURPOSE only.
# Tool usage is driven entirely by the LLM reading tool docstrings.

agent = create_agent(
    "openai:gpt-4o",
    tools=[
        store_user_demographics,
        store_user_preferences,
        get_user_demographics,
        get_user_preferences,
    ],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
    system_prompt="""You are a personalised chef recommendation assistant.
Help users discover recipes, meal ideas, and cooking techniques tailored to their tastes.
The more you learn about a user, the better your recommendations become.
"""
)


# ── Invocation ────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    config = {"configurable": {"thread_id": "prudhvi-chef-session-1"}}

    initial_state = {
        "messages": [{"role": "user", "content": (
            "Hi! I'm Prudhvi from Rajahmundry. "
            "I love spicy food and prefer vegetarian dishes. "
            "Can you recommend something for dinner tonight?"
        )}],
        "demographics": {},
        "preferences": {},
    }

    result = agent.invoke(initial_state, config=config)
    print(result["messages"][-1].content)

It seems there was an issue storing your information. Let me fix that for you.

I'll provide you with a spicy vegetarian dinner recipe recommendation right away. Try making "Spicy Vegetable Stir Fry"! 

**Ingredients:**
- 1 cup broccoli florets
- 1 red bell pepper, sliced
- 1 yellow bell pepper, sliced
- 1 medium carrot, julienned
- 1 cup snow peas
- 2 tablespoons vegetable oil
- 1 tablespoon soy sauce
- 1 tablespoon chili sauce (like Sriracha)
- 2 cloves garlic, minced
- 1 teaspoon grated ginger
- Salt to taste
- Optional garnish: sesame seeds, chopped green onions

**Instructions:**
1. Heat the vegetable oil in a large pan or wok over medium-high heat.
2. Add minced garlic and grated ginger, stir-fry for about 30 seconds until fragrant.
3. Add the broccoli, bell peppers, carrot, and snow peas to the pan. Stir-fry for about 5-7 minutes until the vegetables are just tender but still crisp.
4. Mix in the soy sauce and chili sauce, adding more if you prefer it spicier.
5. Stir well to co